# drfn: grounded routing vs auxiliary heads (deciding grid, seed 11)

Same data, same protocol, same parameter budget (~20.34M) as advisor-update-2019.
The only thing that changes is **where the root/pattern supervision goes**.

| run | labels | supervision target |
|---|---|---|
| `bpe_aux` | real | two linear heads on the final encoder state; output discarded |
| `bpe_placebo` | shuffled | same heads |
| `gmm` | real | the memory's own routing distributions |
| `gmm_placebo` | shuffled | same routing |

A head re-predicts the label independently at each token and throws the answer
away, so it has no way to share anything between two surface forms of one root.
That is why shuffling the labels currently costs only +0.47 chrF++ of a +2.58
gain. Grounded routing makes slot *k* the root class *k*, so every form of a
root reads the same stored vector, and the retrieval feeds the decoder.

The placebo permutes labels **across words**, so under shuffling two forms of a
root land in different slots and the memory cannot share. Read
`(gmm - gmm_placebo)` against `(bpe_aux - bpe_placebo)`.

No analyser and no `geez-morphology` weights are attached: this grid uses the
fixed geezRPA k256 cache shipped in the pack, so it does not drift when a new
morphology run lands.

In [ ]:
SEED = 11
VARIANTS = "bpe_aux bpe_placebo gmm gmm_placebo"

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_grid.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
print('pack', pack_src)
def stream(cmd, env=None, skip_steps=True):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=env)
    for line in p.stdout:
        if skip_steps and (line.startswith('{"step"') or line.startswith("{'step'")):
            continue
        print(line, end='')
    return p.wait()

In [ ]:
env = dict(os.environ, SEED=str(SEED), VARIANTS=VARIANTS)
print('exit', stream(['bash', 'run_grid.sh'], env=env))

In [ ]:
# best dev loss, chrF++ at that epoch, and the routing probe
rows = []
for d in sorted(glob.glob('runs/*_s*')):
    lg = os.path.join(d, 'train_log.jsonl')
    md = os.path.join(d, 'metadata.json')
    if not os.path.exists(lg): continue
    ev = [json.loads(l) for l in open(lg) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    rc = {}
    if os.path.exists(md):
        rc = (json.load(open(md)).get('routing_consistency_root') or {})
    rows.append((os.path.basename(d), len(ev), b['dev_loss'], b['dev_chrf'],
                 b['epoch'], rc.get('consistency'), rc.get('permutation')))
print('%-22s %6s %10s %8s %5s %8s %8s' %
      ('run', 'epochs', 'dev_loss', 'chrF++', 'best', 'route', 'perm'))
for r in rows:
    print('%-22s %6d %10.4f %8.2f %5d %8s %8s' %
          (r[0], r[1], r[2], r[3], r[4],
           'n/a' if r[5] is None else '%.3f' % r[5],
           'n/a' if r[6] is None else '%.3f' % r[6]))

by = {r[0]: r for r in rows}
chrf = lambda n: by[n][3] if n in by else None
route = lambda n: by[n][5] if n in by else None
S = globals().get('SEED', 11)

need = [f'bpe_aux_s{S}', f'bpe_placebo_s{S}', f'gmm_s{S}', f'gmm_placebo_s{S}']
if all(n in by for n in need):
    head = chrf(need[0]) - chrf(need[1])
    mem = chrf(need[2]) - chrf(need[3])
    print('\n--- the deciding comparison ---')
    print('head   real - placebo : %+.2f chrF++   (published value +0.47)' % head)
    print('memory real - placebo : %+.2f chrF++' % mem)
    print('difference of gaps    : %+.2f' % (mem - head))
    print('\nA memory gap clearly larger than the head gap is the result the')
    print('architecture predicts: only the memory can share one stored vector')
    print('across the surface forms of a root, and the placebo permutes labels')
    print('ACROSS WORDS, which destroys exactly that sharing. Equal gaps mean')
    print('morphological CONTENT adds nothing beyond auxiliary-task')
    print('regularization at this data size -- a controlled negative result,')
    print('which is also worth reporting.')

ent = [f'cmm_s{S}_diffuse', f'cmm_s{S}_sharpen', f'cmmspec_s{S}_sharpen']
if any(n in by for n in ent):
    print('\n--- entropy sign and module fidelity ---')
    print('%-26s %8s %8s %8s' % ('run', 'chrF++', 'route', 'perm'))
    for n, what in zip(ent, ['published cmm.py, entropy MAXIMIZED (as shipped)',
                             'published cmm.py, entropy minimized (section 3.6)',
                             'spec 3.4 module, entropy minimized']):
        if n in by:
            r = by[n]
            print('%-26s %8.2f %8s %8s   <- %s' %
                  (n, r[3], 'n/a' if r[5] is None else '%.3f' % r[5],
                   'n/a' if r[6] is None else '%.3f' % r[6], what))
    print('\nPublished reference: routing consistency 0.337, permutation 0.330.')
    print('If the sharpened run moves off chance, the negative result was')
    print('measured under a term pushing routing toward chance. If only the')
    print('spec module moves, the published cmm.py was the problem. If neither')
    print('moves, the mechanism genuinely fails to bind at 13.8K pairs -- and')
    print('that is now a clean negative result rather than a confounded one.')
print('\nNOTE: greedy chrF++ on 256 dev sentences is noisy. This seed-11 grid')
print('is a go/no-go signal, not the thesis number. The six-seed grid and the')
print('held-out paradigm set decide; see DECISION_RULE.')
open('/kaggle/working/summary.txt', 'w').write(repr(rows))